# DATA/MSML 641 - Lecture 4: Sequence Models

© 2020, UNIVERSITY OF MARYLAND  
scienceacademy.umd.edu

## Table of Contents

1. [N-Gram Models](#n-gram-models)
2. [Chain Rule of Probability](#chain-rule-of-probability)
3. [Markov Models](#markov-models)
4. [Language Model Evaluation](#language-model-evaluation)
5. [Information Theory and Cross-entropy](#information-theory-and-cross-entropy)
6. [Surprisal in Psycholinguistics](#surprisal-in-psycholinguistics)
7. [N-gram Length and Performance](#n-gram-length-and-performance)
8. [Probability Estimation and Smoothing](#probability-estimation-and-smoothing)
9. [Hidden Markov Models (HMMs)](#hidden-markov-models)
10. [Conditional Random Fields (CRFs)](#conditional-random-fields)
11. [Decoding and Dynamic Programming](#decoding-and-dynamic-programming)
12. [Named Entity Recognition (NER)](#named-entity-recognition)

## N-Gram Models

### Introduction to N-Gram Models (1/2)

N-gram models represent two different approaches to understanding language:

**Raw form example:** "Yesterday the dog that likes snow chased a squirrel up the tree"

**Linguistic focus:** `{<form, meaning> pairs}`

**Visual representation from slide showing hierarchical structure:**
```
yesterday
├── chasing
    ├── dog
    │   ├── likes
    │   │   └── snow
    └── squirrels
        ├── run
        │   └── squirrels
        └── up tree
```

This represents the syntactic and semantic structure of language with hierarchical relationships between concepts, showing how traditional linguistic analysis focuses on form-meaning pairs and structural relationships.

### N-Gram Models (2/2)

**Language modelling:** Pr(w₁...wₙ)

Key characteristics:
- **Model just the forms, as a sequence**
- **Theoretically:** This is a very different view of language
- **Practically:** Useful for probability and prediction! (See: ASR, MT, auto-suggestion, etc.)

**Examples:**

1. **Prediction task:** "I went swimming at the `<MASK>`." Where did I go swimming?
   - `<MASK>` = beach, peach, pool?

2. **Language identification:** S = "Il n'est pas un professeur". Which language is S written in?
   - Pr_L1(S) > Pr_L2(S) ?

## Chain Rule of Probability

### Remember the Chain Rule of Probability (1/2)

**Basic chain rule:**
```
Pr(a,b,c) = Pr(a) Pr(b|a) Pr(c|a,b)
```

**Applying this formula to tokenized string:**
```
Pr(w₁, w₂, …, wₙ) = Pr(w₁) Pr(w₂|w₁) Pr(w₃ | w₁ w₂) … Pr(wₙ | w₁ … wₙ₋₁)
```

**This generative process can get complex, quick!**
- Pr(w₁) Requires O(V) parameters
- Pr(w₂ | w₁) Requires O(V²) parameters  
- Pr(w₃ | w₁ w₂) Requires O(V³) parameters
- Where V refers to the number of words in the corpus vocabulary

### Remember the Chain Rule of Probability (2/2)

**Observation:** Remember Zipf's law. Most of these word combinations are low-frequency or never-before-seen.

This creates a fundamental problem in language modeling - we need to estimate probabilities for combinations we've rarely or never seen before.

## Markov Models

### Markov Model (1/2)

**Markov model assumes the Markov property**

- **1st order Markov:** Pr(wᵢ | w₁ … wᵢ₋₁) is presumed equal to Pr(wᵢ | wᵢ₋₁)
- **Bigram model = 1st-order Markov approximation**
- **Effectively simplifying the problem** by defining the previous word as an equivalence class of all histories ending with that word

### Markov Model (2/2)

**Therefore:**

```
Pr_bigram(w₀w₁w₂⋯wₙ) = ∏ᵢ₌₁ⁿ Pr(wᵢ | wᵢ₋₁)
```

**In log space:**
```
log Pr(w₁⋯wₙ) = ∑ log Pr(wₙ | wᵢ₋₁)
```

**Benefits:**
- **Log avoids underflow**
- **Addition more efficient than multiplication**

**To make life easier we assume w₀ is a special START symbol such that Pr(w₀) = 1.**

## Language Model Evaluation

### Evaluation of Language Models (1/2)

**What makes a language model (LM) good?** It gives high probability to the test set.

- **Goal** is to estimate Pr_LM from training data, and evaluate the LM by feeding it to unseen test data W
- **Perplexity:** how "surprised" or "perplexed" is the model when it sees the test data?
- **Lower perplexity is better**

### Evaluation of Language Models (2/2)

**Perplexity formulas:**

```
PPₘ(w) = ᴺ√(1/Pₘ(w))    where w = w₁⋯wₙ
```

```
PP_bigram(w) = ᴺ√(∏ᵢ₌₁ᴺ 1/Pr(wᵢ | wᵢ₋₁))
```

**Key insights:**
- **Can apply perplexity per word, then average at the end to reduce impact of length bias**
- **Also interpretable as the avg number of choices per prediction**

### Exercise: Perplexity Calculation

#### Exercise (Slide 11)
**Problem:**
- Assume your vocabulary has 10 words: (zero, one, two, …, nine)
- Suppose LM is a uniform distribution. What is PP(W)?
- Hint: you don't need to know what W is

#### Solution (Slide 12)
**Answer:**
```
PP(W) = (1/10 ^ N )^(-1/N) = 10
```

**Step-by-step explanation:**

For a uniform distribution over V words:
1. Each word has probability P(wᵢ) = 1/V = 1/10
2. For any sequence of length N: P(w₁...wₙ) = (1/V)ᴺ = (1/10)ᴺ  
3. Perplexity: PP(W) = ᴺ√(1/P(W)) = ᴺ√(1/(1/10)ᴺ) = ᴺ√(10ᴺ) = 10

**Interpretation:**
- For a uniform distribution over V words, perplexity always equals V
- PP(W) = 10 means the model is effectively choosing among 10 equally likely options
- This is the baseline perplexity for a uniform language model

## Information Theory and Cross-entropy

### An Information-theoretic View: How Perplexity Relates to Cross-entropy (1/4)

**Entropy** measures uncertainty (surprise)

```
H(x) = -∑ₓ p(x) log₂ p(x)    for x ~ p
```

**Example: Coin flip entropy**
- Entropy H varies from 0 to 1 as Pr(Heads) goes from 0 to 0.5 to 1
- Maximum entropy (uncertainty) occurs at Pr(Heads) = 0.5
- Minimum entropy occurs at the extremes (certainty)

**Question:** How uncertain is this coin flip?

### An Information-theoretic View: How Perplexity Relates to Cross-entropy (2/4)

**Horse race betting example:**

Placing a bet on a horse race, it is too far and you want to send **SHORT** messages to the bookie. Suppose there are 8 horses.

**Sending SHORT messages:**

**Solution #1:** Binary representation of horse numbers as code:
- 1→ 001, 2→ 010, 3→ 011, …, 8→ 000
- Every time 3 bits need to be sent!!!

**Solution #2 (better way):** Use actual distribution of bets as prior probabilities:
- H #1: 1/2, H #2: 1/4, H #3: 1/8, H #4: 1/16, H #5,6,7,8: 1/64

**Lower-entropy distributions require fewer bits to encode outcomes**

### An Information-theoretic View: How Perplexity Relates to Cross-entropy (2/4 continued)

**Comparing encoding schemes:**

**P1 = (1/8, 1/8, 1/8, 1/8, 1/8, 1/8, 1/8, 1/8)**
- H(P1) = 3 bits (000, 001, 010, …, 111) on average; same for every outcome

**P2 = (1/2, 1/4, 1/8, 1/16, 1/64, 1/64, 1/64, 1/64)**
- H(P2) = 2 bits on average (0, 10, 110, 1110, 111100, 111101, 111110, 111111)
- **More probable outcomes get shorter codes**

### An Information-theoretic View: How Perplexity Relates to Cross-entropy (3/4)

**Cross-entropy measures the quality of prediction**

- Let **p** be the "true" language model
- Let **m** be a model (i.e., an approximation of p): pr_m(wᵢ | w₁ … wᵢ₋₁)

**Under certain assumptions:**
```
H(p,m) = lim(n→∞) -1/n ∑w p(W₁, W₂, …Wₙ) log m(W₁, W₂, …, Wₙ)
```

**In practice, we estimate as:**
```
Hₘ(w) = -1/N log m(w₁⋯wₙ)
       = -1/N log ∏P(wᵢ | w₁⋯wᵢ₋₁)  
       = -1/N ∑ log P(wᵢ | w₁⋯wᵢ₋₁)
```

Where the sequence w is drawn from P (the true distribution).

**Note:** The sequence w in the practical estimation is drawn from the true distribution P, which allows us to estimate the cross-entropy between the true model p and our approximation m.

### An Information-theoretic View: How Perplexity Relates to Cross-entropy (3/3)

**In practice, we estimate as:**
```
Hₘ(w) = -1/N log m(w₁⋯wₙ)
       = -1/N log ∏P(wᵢ | w₁⋯wᵢ₋₁)
       = -1/N ∑ log P(wᵢ | w₁⋯wᵢ₋₁)
```

**One can show:**
```
PPₘ(w) = ᴺ√(1/Pₘ(w)) = 2^Hₘ(w)
```

**Cross entropy measures the same thing as perplexity – how well the model predict the data – but in bits.**

## Surprisal in Psycholinguistics

### Surprisal (1/2)

A related measure in psycholinguistics, the **surprisal** at a word wᵢ in the input can be characterized as:

```
-log Pₘ(wᵢ | context)
```

This correlates with measures of cognitive work being done during human sentence processing.

### Surprisal (2/2)

**Example:**
"I drink my coffee with cream and **socks** in the morning when I get up"

In response to unexpected words like "socks", you can expect to measure:
- **Longer reading times** in eye tracking experiments
- **Event related potentials** (e.g. N400 or P600) using EEG
- **Increased blood oxygen level** using fMRI

This demonstrates how computational language models can predict human cognitive processing difficulty.

## N-gram Length and Performance

### Increasing N-gram Length = Better Language Models?

**Yes, up to a point:**

- **More context provides more accurate conditional probabilities**
- **But at some point sparsity becomes a problem**
- **Recall Zipf:** a trigram model has O(V³) parameters, and most counts will be 0!!!!

This creates the fundamental trade-off in n-gram modeling between capturing longer dependencies and having sufficient data to estimate parameters reliably.

## Probability Estimation and Smoothing

### Probability Estimation for LMs

**The problem of zeros**
- Suppose a LM predicts wᵢ has zero probability after w₁ … wᵢ₋₁
- What's the perplexity? → ∞ (undefined)

**Smoothing methods ensure probabilities are never zero**
- They "steal" probability mass from frequent n-grams to distribute to unseen n-grams

**Example: Laplace smoothing (add-1 to all probabilities)**
- Initialize the count to 1 for each word, so we start from 1 (instead of 0)
- P(wᵢ) = (Count_wᵢ + 1) / (N+V), where V = size of entire vocabulary
- **Add-k:** Laplace adding k instead of 1 (e.g. k=1/V)

### Interpolated Smoothing

**Interpolated smoothing formula:**
```
P̂(wₙ | wₙ₋₂ wₙ₋₁) = λ₃P(wₙ | wₙ₋₂ wₙ₋₁)
                    + λ₂P(wₙ | wₙ₋₁)
                    + λ₁P(wₙ)
```
subject to: ∑λᵢ = 1

**Key insights:**
- **Ideally, λ₃ will be high when the trigram model is a good predictor**
- **If too little data, λ₃ is lower, so more weight will be available for λ₁ and λ₂**
- **Hence greater reliance on the better-estimated unigrams and bigrams**

### N-gram Tools and Resources

**"All Our N-gram are Belong to You"** (Reference to classic internet meme)

Popular N-gram modeling toolkits:

**KenLM**
- https://kheafield.com/code/kenlm/

**SRILM**  
- http://www.speech.sri.com/projects/srilm/

**Historical Reference: Google N-Gram Release, August 2006**

The slide shows a reference to Google's massive N-gram dataset release with the title "All Our N-gram are Belong to You" posted by the Google Research team. This was a significant release that provided:

- Enormous datasets of N-grams for a variety of NLP projects
- Counts for 1-grams through 5-grams extracted from a large web corpus
- Over 1 trillion tokens and over 13 million unique words
- This resource was foundational for many NLP research projects

The Google N-gram datasets served as the incoming 52 terms as the incubator 49 terms as the index 233, etc., providing crucial frequency statistics for language modeling research.

## Hidden Markov Models

### Sequence Modeling: HMMs

**There are better sequential models for language:**
- Conditional random fields
- Structured Perceptrons  
- Neural sequence models (current state of the art)

**But HMMs bring together multiple key ideas in NLP:**
- **Latent generative models**
- **Dynamic programming**
- **Noisy channel models**

### Noisy Channel Models (NCM)

```
T = T₁ …. Tₙ    →    [Source]    →    [Channel]    →    W = w₁ …. wₙ    →    T = ?
                     Pr(T)           Pr(W | T)                              [Decoder]
```

**Components:**
- **Source:** Generates hidden sequence T
- **Channel:** Generates observed sequence W given T
- **Decoder:** Recovers T from observed W

**NCM is a framework used in:**
- Spell checking
- QA (Question Answering)
- Speech recognition
- Machine translation

### Noisy Channel Models - Mathematical Formulation

**Complete joint probability from the slide:**
```
Pr(t₁⋯tₙ) = ∏ᵢ Pr(tᵢ | tᵢ₋₁) ∏ᵢ Pr(wᵢ | tᵢ)
```

**Components breakdown:**
- **Bigram model:** Pr(tᵢ | tᵢ₋₁) - Transition probabilities
- **Substitution not conditioned on context:** Pr(wᵢ | tᵢ) - Emission probabilities

**Therefore:**
```
Pr(T,W) = Pr(t₁⋯tₙ, w₁⋯wₙ)
        = ∏ᵢ Pr(tᵢ|tᵢ₋₁) Pr(wᵢ | tᵢ)
```

**Terminology from slide:**
- **Latent/'hidden' from decoder:** T (the tag sequence)
- **Observed by decoder:** W (the word sequence)  
- **Markov:** N-gram model over hidden variables
- **N-gram model over hidden variables:** The transition probabilities follow Markov assumptions

This formulation combines:
1. A generative model for the hidden state sequence (Markov chain)
2. An observation model that generates words from states (emission probabilities)

### HMM As a Hidden-state Machine (1/2)

**Process:**
- **Source generates "hidden states" (tags)**
- **Channel generates observed symbols**
- **How to recover the "hidden states"?**
- **Use both transition probabilities and emission probabilities to estimate the probability of the observed sequence**

**Example:**
```
T = DET Adj N V ….    →    W = the brown dog barked ….
```

**Vocabularies:**
- **Source:** Vocabulary of "hidden states"
- **Channel:** Vocabulary of "observed symbols"

### HMM As a Hidden-state Machine (2/2)

**State transition diagram showing:**
- States: `<s>`, DET, Adj, N, V
- Transitions between states with probabilities

**Two probability matrices:**

**A = Pr(tⱼ | tᵢ) - Transition probabilities**
```
        <s>  Det  Adj   N    V
START    1    0    0    0   ...   0
DET      0   .2   .2   .03  ...   0
Adj      0
N        0
V        0
```

**B = Pr(wₖ | tᵢ) - Emission probabilities**
```
States emit words: <s> → a, an, another, ..., zoo
```

### HMM as Graphical Model

**Graphical representation:**
```
T₁ → T₂ → T₃ → ... → Tₙ
↓    ↓    ↓          ↓
W₁   W₂   W₃        Wₙ
```

**Key properties:**
- **Arrows represent conditional dependence**
- **Each Tⱼ depends only on the previous Tⱼ₋₁**
- **Each Wᵢ depends only on Tᵢ**

This is a directed graphical model showing the Markov assumption.

## Conditional Random Fields

### CRF As a Graphical Model

**Linear-chain Conditional Random Field**

```
Y = y₁ — y₂ — y₃ — ... — yₙ
     ∖   |   /  ⋱    /
       ∖ | /  ⋱   /
         X = X₁ X₂ ….... Xₙ
```

**Mathematical formulation:**
```
P(Y | X; w̄) = exp(w̄ · F̄(X,Y)) / ∑ᵧ exp(w̄ · F̄(X,Y))
```

where
```
Fⱼ(X,Y) = ∑ᵢ Fⱼ(yᵢ₋₁yᵢ, X, i)
```

**Key characteristics:**
- **Discriminative, not generative**
- **Undirected graph**
- **Log-linear model**
- **Entire observed X available to feature functions Fⱼ**
- **Limited pieces of latent Y available at a time to the feature functions: Markov assumption**

## Decoding and Dynamic Programming

### Decoding Using HMMs

**Decoding = search, guided by a model**

**But how do you compute P(T|W)? With Bayes!**

```
Goal = argmaxₜ Pr(W | T) Pr(T) / Pr(W)
     = argmaxₜ Pr(W | T) Pr(T) / Pr(W)
```

Since Pr(W) is not relevant in argmax over T:
```
argmaxₜ Pr(T | W) = argmaxₜ Pr(W | T) Pr(T)
```

**Components:**
- **We have the model, HMM = <A, B, Pi>**
- **Now we just need to search through all possibilities T = t₁ … tₙ**
- **… But that's an exponential number of possibilities.**

### Decoding HMMs–Dynamic Programming (1/2)

**Forward probability computation:**

```
        Hidden
        symbol   a₁  O  O ⋯  O   O   O ⋯   O
                 a₂  O  O ⋯  O   O   O ⋯   O
                 ⋮   ⋮  ⋮      ⋮   ⋮   ⋮
        Observed aₙ  O  O     O   O   O     O
        symbol       O₁ O₂ ⋯ Oₜ₋₁ Oₜ Oₜ₊₁ ⋯ Oₜ
                    <S>                    
```

**What's the probability of a path going through state aⱼ at time t?**

**αₜ(j) is known as the "forward" probability.**

**Base case:**
```
α₁(START) = 1 = probability of starting at START, emitting <S>
```

**Recursive case:**
```
αₜ(j) = ∑ᵢ₌₁ᴺ αₜ₋₁(i)aᵢⱼbⱼ(Oₜ)
```

### Decoding HMMs–Dynamic Programming (2/2)

**Goal:** determine the highest probability path of hidden states a₁… aₜ  
**generating observed symbols O₁…Oₜ, given HMM <A, B, Pi>**

**Key question: What's the probability of one path?**

**Complete probability calculation from slide:**
```
Pr(S₀) × Pr(S₁ | S₀)Pr(O₁ | S₁) × Pr(S₂ | S₁)Pr(O₂ | S₂) × Pr(S₃ | S₂)Pr(O₃ | S₃) × ⋯
```

**Breaking down the components:**
- **= Π(S₀)** = probability of initial state
- **= Π(<S>)** = probability of START symbol  
- **= 1** (by definition, we always start at START)

**Notation clarification:**
- **Transition probability in A:** aS₁S₂ (from state S₁ to S₂)
- **Emission probability in B:** bS₂(O₂) (state S₂ emitting observation O₂)

**General formula for any path:**
```
Path probability = π(s₁) × ∏ᵢ₌₁ᵀ as_{i-1},s_i × ∏ᵢ₌₁ᵀ bs_i(oᵢ)
```

Where:
- π(s₁) = initial state probability
- as_{i-1},s_i = transition probability from state sᵢ₋₁ to sᵢ  
- bs_i(oᵢ) = emission probability of observation oᵢ from state sᵢ

### Probabilities of a Path at Time t (1/2)

**Forward algorithm visualization:**

```
START ——————————————————————————————————→
      ↘   O   O ⋯ O  αₜ(1)   ∑ᵢ₌₁ⁿ
        ↘   ⋮   ⋮     ⋮  αₜ(2)
          ↘ ⋮   ⋮     ⋮   ⋮
            O   O     O  αₜ(N)
           <S> O₁     Oₜ
```

**Key insights:**
- **Probability of any path ending at state a₁**
- **Sum of probabilities of all possible paths ending in state i**
- **= Sum of probabilities of all hidden generating paths for O₁ …… Oₜ**
- **= ∑ₛ₁,…,ₛₜ Pr(S₁,⋯,Sₙ, O₁,⋯,Oₜ)**
- **= Pr(O₁,⋯,Oₜ)**

**Probabilities of all paths computable in polynomial time thanks to Dynamic Programming (DP)**

### Probabilities of a Path at Time t (2/2)

### Viterbi Decoding

**What is the probability of the highest-probability path?**

**Just replace the function: summation with the max!**

Instead of:
```
αₜ(j) = ∑ᵢ₌₁ᴺ αₜ₋₁(i)aᵢⱼbⱼ(Oₜ)
```

Use:
```
δₜ(j) = maxᵢ₌₁ᴺ δₜ₋₁(i)aᵢⱼbⱼ(Oₜ)
```

**At every step in the DP algorithm, we are extending with the best probability, rather than the total probability.**

### Estimating HMM Probabilities (1/2)

**Two main approaches:**

**Supervised methods:** tagged corpus → smoothed counts
- When we have labeled training data
- Directly estimate transition and emission probabilities from counts

**Unsupervised methods:**
- **Select number of states N**
- **Use Expectation Maximization (EM)**
- **Interpret the meaning of hidden states**

### Estimating HMM Probabilities (2/2)

**Example with N = 5 states:**

**State transition diagram:**
```
S₁ → S₂
↓ ↗  ↓
S₅ ← S₃
↑    ↓
S₄ ←─┘
```

**Emission probability table:**
```
     S1   S2   S3   S4   S5
<S>  the  big  man  hit
.    a    little dog jumped
.    an   blue people .
.    one  .    .    .
.    some .    .    .
.    .    .    .    .
```

**Interpretations:**
- **High probability aᵢⱼ:** Strong transition preferences
- **High probability bⱼₖ:** Words strongly associated with states

### More on Unsupervised Estimation of HMM Probabilities

**Analogous to data clustering**

**Latent clusters visualization:**
```
    •   • •
  •   •  ●  •    ←─ Latent clusters
     • •   •
  ●     •  •  ●
   •  • •    •
     • •  ●
```

**Process:**
- **Interpretation: look at items closest to centroid**
- **Example: latent state S3 probably corresponds to the category "adjective"**

This unsupervised learning discovers syntactic categories without explicit linguistic knowledge.

## Named Entity Recognition

### Named Entity Recognition (NER)

**Classic inventory of "named entities" includes categories like:**
- **People**
- **Organizations**
- **Locations**
- **Geopolitical entities**

**Some related problems:**
- **Identifying text spans that are named entities**
- **Labelling identified text spans with a type (e.g. LOC for locations)**
- **Entity linking (e.g. {Biden, Joe Biden, President Biden})**

### NER Evaluation

**Goal:** identify ("retrieve") all and only the correct ("relevant") entities

**Standard metrics:**
- **Precision = (relevant and retrieved) / retrieved**
- **Recall = (relevant and retrieved) / relevant**

These metrics come from information retrieval and are widely used in NLP evaluation.

### Entity Recognition As a Sequence Labeling Task

**Tag words with B, I, O (beginning, inside, outside – or variations thereof)**

**Example 1: Basic BIO tagging**
```
Jane Villanova of United Airlines Group discussed the Chicago flights
B    I         O  B      I        I     O         O   B       O
```

**Example 2: Extending to entity type**
Tag set: `{B-t, I-t} ∪ {O}`, where t is any of {PER, LOC, ORG, …}

```
Jane Villanova of United Airlines Group discussed the Chicago flights
[B-PER I-PER] O [B-ORG I-ORG I-ORG] O O [B-LOC] O
```

This transforms NER into a sequence labeling problem suitable for HMMs, CRFs, or neural models.

## Summary

This lecture covered the fundamental concepts of sequence modeling in NLP:

1. **N-gram models** as the foundation for statistical language modeling
2. **Markov assumptions** to make the modeling tractable
3. **Evaluation metrics** like perplexity and cross-entropy
4. **Smoothing techniques** to handle sparse data
5. **Hidden Markov Models** for modeling latent structure
6. **Dynamic programming algorithms** for efficient inference
7. **Applications** to practical NLP tasks like Named Entity Recognition

These concepts form the basis for understanding more advanced sequence models including neural approaches that have become dominant in modern NLP.